# XGBoost Classification

This notebook builds an XGBoost classifier to predict the **Category** of a product purchase. We mirror the structure of the Random Forest notebook ([09-random-forest.ipynb](09-random-forest.ipynb)) so results are directly comparable.

**Key constraints (same as prior models):**
- 1,625 unique product categories (extreme multi-class)
- Full 157K training set (32GB RAM, `tree_method='hist'`)
- Temporal train/test split (train ≤ 2021, test > 2021)

**Progress benchmarks:**
- Random Forest (tuned): ~5.5% accuracy
- Wide & Deep NN (tuned): ~6.4% accuracy

## 1. Data Loading and Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

# Scorer with zero_division=0 prevents NaN when folds contain unseen classes
scorer = make_scorer(f1_score, average='weighted', zero_division=0)

print(f'XGBoost version: {xgb.__version__}')

In [ ]:
# ── GPU CONFIG ────────────────────────────────────────────
# True  → uses RTX 4060 via CUDA (faster, needs 8GB VRAM)
# False → uses all CPU cores (n_jobs=-1), safe for everyone
USE_GPU = True

xgb_device = 'cuda' if USE_GPU else 'cpu'
xgb_njobs  = 1 if USE_GPU else -1

print(f'Device: {xgb_device} | n_jobs: {xgb_njobs}')
# ─────────────────────────────────────────────────────────

In [ ]:
data = pd.read_csv('../data/cleaned_data.csv')
print(f'Dataset shape: {data.shape}')
print(f'Unique categories: {data["Category"].nunique()}')

In [ ]:
# Temporal train/test split
train = data[data['order_year'] <= 2021].copy()
test = data[data['order_year'] > 2021].copy()

# Drop test samples with categories unseen during training
train_cats = set(train['Category'].unique())
test_cats = set(test['Category'].unique())
unseen = test_cats - train_cats
test = test[~test['Category'].isin(unseen)]

print(f'Train: {len(train):,} | Test: {len(test):,} (removed {len(unseen)} unseen categories)')

In [ ]:
# Drop non-predictive columns and split features/target
drop_cols = ['Title', 'ASIN/ISBN (Product Code)']

X_train = train.drop(['Category'] + drop_cols, axis=1)
y_train_raw = train['Category']

X_test = test.drop(['Category'] + drop_cols, axis=1)
y_test_raw = test['Category']

# XGBoost requires contiguous integer labels [0, num_class)
le = LabelEncoder()
le.fit(y_train_raw)
y_train = le.transform(y_train_raw)
y_test = le.transform(y_test_raw)

num_classes = len(le.classes_)
print(f'Features: {X_train.shape[1]} | Classes: {num_classes}')

In [ ]:
# Scale features (consistent with RF notebook — tree models don't require it but keeps comparability)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Use full training set — 32GB RAM handles 157K × 156 comfortably with tree_method='hist'
X_train_sub = X_train_scaled
y_train_sub = y_train

print(f'Train: {X_train_sub.shape} | Test: {X_test_scaled.shape}')

## 2. Baseline XGBoost

Using `tree_method='hist'` for memory efficiency. Device and parallelism are controlled by the `USE_GPU` flag above.

In [ ]:
%%time

# XGBoost multi-class builds num_class trees per round.
# With 1,625 classes, n_estimators=50 produces ~81,250 trees — hist keeps this memory-efficient.
xgb_baseline = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=num_classes,
    tree_method='hist',
    device=xgb_device,
    n_jobs=xgb_njobs,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_baseline.fit(X_train_sub, y_train_sub)
print('Baseline XGBoost trained.')

In [ ]:
y_pred_baseline = xgb_baseline.predict(X_test_scaled)

baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_f1_macro = f1_score(y_test, y_pred_baseline, average='macro', zero_division=0)
baseline_f1_weighted = f1_score(y_test, y_pred_baseline, average='weighted', zero_division=0)

print(f'Baseline Accuracy:         {baseline_acc:.4f}')
print(f'Baseline F1 (macro):       {baseline_f1_macro:.4f}')
print(f'Baseline F1 (weighted):    {baseline_f1_weighted:.4f}')

## 3. Feature Importance Analysis

In [ ]:
feature_names = X_train.columns
importances = xgb_baseline.feature_importances_

feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print('Top 20 most important features:')
feat_imp.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top_n = 30
top_feats = feat_imp.head(top_n)

ax.barh(range(top_n), top_feats['Importance'].values, color='darkorange')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_feats['Feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('Top 30 Feature Importances â€” XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
feat_imp['Cumulative'] = feat_imp['Importance'].cumsum()

n_90 = (feat_imp['Cumulative'] <= 0.90).sum() + 1
n_95 = (feat_imp['Cumulative'] <= 0.95).sum() + 1

print(f'Features needed for 90% importance: {n_90} / {len(feature_names)}')
print(f'Features needed for 95% importance: {n_95} / {len(feature_names)}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(feat_imp) + 1), feat_imp['Cumulative'].values, color='darkorange')
ax.axhline(y=0.90, color='r', linestyle='--', label='90%')
ax.axhline(y=0.95, color='steelblue', linestyle='--', label='95%')
ax.set_xlabel('Number of Features')
ax.set_ylabel('Cumulative Importance')
ax.set_title('Cumulative Feature Importance â€” XGBoost')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning

`RandomizedSearchCV` over XGBoost's key hyperparameters. Each candidate uses `n_jobs=-1` internally; the outer search runs sequentially (`n_jobs=1`) to avoid spawning nested parallelism.

In [ ]:
%%time

param_distributions = {
    'n_estimators': [30, 50, 100],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3]
}

xgb_search = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    tree_method='hist',
    device=xgb_device,
    n_jobs=xgb_njobs,
    random_state=42,
    eval_metric='mlogloss'
)

random_search = RandomizedSearchCV(
    xgb_search,
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring=scorer,
    random_state=42,
    n_jobs=1,
    verbose=2
)

random_search.fit(X_train_sub, y_train_sub)

print(f'\nBest score (F1 weighted): {random_search.best_score_:.4f}')
print(f'Best parameters: {random_search.best_params_}')

In [ ]:
search_results = pd.DataFrame(random_search.cv_results_)
search_results = search_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
search_results.sort_values('rank_test_score').head(10)

## 5. Tuned Model 

In [ ]:
%%time

best_params = random_search.best_params_
print(f'Training tuned model with: {best_params}')

xgb_tuned = xgb.XGBClassifier(
    **best_params,
    objective='multi:softprob',
    num_class=num_classes,
    tree_method='hist',
    device=xgb_device,
    n_jobs=xgb_njobs,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_tuned.fit(X_train_sub, y_train_sub)
print('Tuned model trained.')

In [ ]:
y_pred_tuned = xgb_tuned.predict(X_test_scaled)

tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_prec_macro = precision_score(y_test, y_pred_tuned, average='macro', zero_division=0)
tuned_rec_macro = recall_score(y_test, y_pred_tuned, average='macro', zero_division=0)
tuned_f1_macro = f1_score(y_test, y_pred_tuned, average='macro', zero_division=0)
tuned_f1_weighted = f1_score(y_test, y_pred_tuned, average='weighted', zero_division=0)

print('=== Tuned XGBoost Results ===')
print(f'Accuracy:             {tuned_acc:.4f}')
print(f'Precision (macro):    {tuned_prec_macro:.4f}')
print(f'Recall (macro):       {tuned_rec_macro:.4f}')
print(f'F1 Score (macro):     {tuned_f1_macro:.4f}')
print(f'F1 Score (weighted):  {tuned_f1_weighted:.4f}')

print(f'\n--- Comparison ---')
print(f'Baseline XGB accuracy: {baseline_acc:.4f}')
print(f'Tuned XGB accuracy:    {tuned_acc:.4f}')
print(f'Improvement:           {tuned_acc - baseline_acc:+.4f}')

In [ ]:
# Classification report for top 20 most frequent categories
y_test_series = pd.Series(y_test)
top_20_cats = y_test_series.value_counts().head(20).index.tolist()
mask = y_test_series.isin(top_20_cats)

print('Classification Report â€” Top 20 Most Frequent Categories:')
print(classification_report(
    y_test[mask], y_pred_tuned[mask],
    labels=top_20_cats,
    zero_division=0
))

In [ ]:
# Confusion matrix for top 15 categories
top_15_cats = y_test_series.value_counts().head(15).index.tolist()
mask_15 = y_test_series.isin(top_15_cats)

# Map encoded labels back to original category IDs for readability
original_top_15 = le.inverse_transform(top_15_cats)

cm = confusion_matrix(y_test[mask_15], y_pred_tuned[mask_15], labels=top_15_cats)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=original_top_15, yticklabels=original_top_15, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix â€” Top 15 Categories (XGBoost)')
plt.tight_layout()
plt.show()

## 6. Cross-Validation

In [ ]:
%%time

train_df = pd.DataFrame(X_train_sub, columns=feature_names)
train_df['Category'] = y_train_sub

cat_counts = train_df['Category'].value_counts()
valid_cats = cat_counts[cat_counts >= 5].index
train_df_filtered = train_df[train_df['Category'].isin(valid_cats)]

le_cv = LabelEncoder()
X_cv = train_df_filtered.drop('Category', axis=1).values
y_cv = le_cv.fit_transform(train_df_filtered['Category'].values)
num_classes_cv = len(le_cv.classes_)

print(f'CV data: {len(X_cv):,} samples, {num_classes_cv} categories (filtered to >= 5 per class)')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_cv = xgb.XGBClassifier(
    **best_params,
    objective='multi:softprob',
    num_class=num_classes_cv,
    tree_method='hist',
    device=xgb_device,
    n_jobs=xgb_njobs,
    random_state=42,
    eval_metric='mlogloss'
)

cv_scores = cross_val_score(xgb_cv, X_cv, y_cv, cv=cv, scoring=scorer, n_jobs=1)

print(f'\nCV F1 (weighted) scores: {cv_scores}')
print(f'Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})')

In [ ]:
cv_acc_scores = cross_val_score(xgb_cv, X_cv, y_cv, cv=cv, scoring='accuracy', n_jobs=1)

print(f'CV Accuracy scores: {cv_acc_scores}')
print(f'Mean: {cv_acc_scores.mean():.4f} (+/- {cv_acc_scores.std() * 2:.4f})')

## 7. Summary

In [ ]:
print('=' * 55)
print('          XGBOOST — RESULTS SUMMARY')
print('=' * 55)
print(f'  Dataset:              {data.shape[0]:,} samples, {X_train.shape[1]} features')
print(f'  Categories:           {num_classes}')
print(f'  Train size:           {len(y_train_sub):,}')
print(f'  Test set:             {len(y_test):,}')
print(f'')
print(f'  --- Baseline XGBoost ---')
print(f'  Accuracy:             {baseline_acc:.4f}')
print(f'  F1 (weighted):        {baseline_f1_weighted:.4f}')
print(f'')
print(f'  --- Tuned XGBoost ---')
print(f'  Best params:          {best_params}')
print(f'  Accuracy:             {tuned_acc:.4f}')
print(f'  Precision (macro):    {tuned_prec_macro:.4f}')
print(f'  Recall (macro):       {tuned_rec_macro:.4f}')
print(f'  F1 (macro):           {tuned_f1_macro:.4f}')
print(f'  F1 (weighted):        {tuned_f1_weighted:.4f}')
print(f'')
print(f'  --- Cross-Validation (5-fold, full train) ---')
print(f'  F1 (weighted):        {cv_scores.mean():.4f} +/- {cv_scores.std() * 2:.4f}')
print(f'  Accuracy:             {cv_acc_scores.mean():.4f} +/- {cv_acc_scores.std() * 2:.4f}')
print(f'')
print(f'  --- Model Comparison ---')
print(f'  Random Forest (tuned):    ~5.5% accuracy')
print(f'  Wide & Deep NN (tuned):   ~6.4% accuracy')
print(f'  XGBoost (tuned):          {tuned_acc*100:.1f}% accuracy')
print('=' * 55)